# Post-training — Analyse des résultats (5000 audiogrammes)

Ce notebook charge les résultats produits par `python main.py --data data/`  
Il ne réentraîne rien — il lit `results/abs/scores.csv` et les fichiers JSON sources.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.features import build_feature_matrix, STANDARD_FREQS
from src.evaluate import plot_audiogram

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

RESULTS_DIR = Path('../results/abs')
DATA_PATH   = Path('../data/clean_dataset.json')   # produit par 04_data_cleaning.ipynb

VISIT_LABEL  = {0: 'Baseline', 1: 'Periodic', 2: 'Depart'}
GENDER_LABEL = {1: 'Femme', 2: 'Homme'}            # encodage Odyo

In [ ]:
# ── Scores ────────────────────────────────────────────────────────────────────
scores_df = pd.read_csv(RESULTS_DIR / 'scores.csv', index_col=0)

# ── Dataset propre (même source que l'entraînement) ───────────────────────────
df = pd.read_json(DATA_PATH, orient='records')
for col in ('dots_left', 'dots_right'):
    df[col] = df[col].apply(lambda d: {float(k): v for k, v in d.items()} if isinstance(d, dict) else {})
df['visit_date'] = pd.to_datetime(df['visit_date'], utc=True, errors='coerce')

# ── Features (même pipeline qu'à l'entraînement) ─────────────────────────────
feature_df, _ = build_feature_matrix(df)

# ── Alignement ────────────────────────────────────────────────────────────────
assert len(df) == len(scores_df), (
    f"Désalignement : {len(df)} records vs {len(scores_df)} scores.\n"
    "Relancer main.py --data data/clean_dataset.json pour regénérer les scores."
)
scores_df.index   = df.index
feature_df.index  = df.index

print(f"Dataset  : {len(df)} records — {df['patient'].nunique()} patients")
print(f"Scores   : {scores_df.shape}")
print(f"Features : {feature_df.shape}")
print(f"\nCatégories :")
print(df['visit_category'].map(VISIT_LABEL).value_counts().to_string())
print(f"\nGenre :")
print(df['gender'].map(GENDER_LABEL).value_counts().to_string())

## 1. Résumé global des flags

In [ ]:
flag_cols = ['anomaly_flag_if', 'anomaly_flag_ae', 'anomaly_flag_pca',
             'anomaly_consensus', 'nihl_flag', 'meniere_flag', 'sts_flag', 'anomaly_final']
flag_cols = [c for c in flag_cols if c in scores_df.columns]

summary = pd.DataFrame({
    'n_flaggés': [scores_df[c].fillna(0).sum() for c in flag_cols],
    '% total':   [100 * scores_df[c].fillna(0).mean() for c in flag_cols],
}, index=flag_cols).round(1)

print(summary.to_string())

# Plages épidémiologiques attendues
expected = {'nihl_flag': (15, 30), 'meniere_flag': (1, 5), 'sts_flag': (5, 15)}
print("\nCalibration vs épidémiologie attendue :")
for col, (lo, hi) in expected.items():
    if col in scores_df.columns:
        rate = 100 * scores_df[col].fillna(0).mean()
        status = '✓' if lo <= rate <= hi else '⚠'
        print(f"  {status} {col:20s} {rate:.1f}% (attendu {lo}–{hi}%)")

## 2. Analyse par fichier source

Pour choisir quel fichier labelliser en priorité.

In [ ]:
flag_cols = [c for c in ['nihl_flag', 'meniere_flag', 'anomaly_consensus', 'anomaly_final']
             if c in scores_df.columns]

scores_aug = scores_df.copy()
scores_aug['visit_category'] = df['visit_category'].map(VISIT_LABEL).values

table = scores_aug.groupby('visit_category')[flag_cols + ['reconstruction_error']].agg(
    **{col: (col, lambda x: f"{x.fillna(0).sum():.0f} ({100*x.fillna(0).mean():.1f}%)")
       for col in flag_cols},
    score_moyen=('reconstruction_error', lambda x: f"{x.mean():.3f}"),
)
print(table.to_string())

In [ ]:
flag_cols = [c for c in ['nihl_flag', 'meniere_flag', 'anomaly_consensus', 'anomaly_final']
             if c in scores_df.columns]

scores_aug = scores_df.copy()
scores_aug['visit_category'] = df['visit_category'].map(VISIT_LABEL).values

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Taux de flags par catégorie
cat_rates = scores_aug.groupby('visit_category')[flag_cols].apply(
    lambda g: g.fillna(0).mean() * 100
)
cat_rates.T.plot(kind='bar', ax=axes[0], edgecolor='white', alpha=0.85)
axes[0].set_ylabel('% flaggés')
axes[0].set_title('Taux de flags par catégorie de visite')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right')
axes[0].grid(axis='y', alpha=0.3)
axes[0].legend(title='Catégorie', fontsize=8)

# Distribution du score par catégorie
if 'reconstruction_error' in scores_aug.columns:
    palette = {'Baseline': 'steelblue', 'Periodic': 'darkorange', 'Depart': 'tomato'}
    for cat, grp in scores_aug.groupby('visit_category'):
        axes[1].hist(grp['reconstruction_error'].dropna(), bins=40, alpha=0.6,
                     label=f'{cat} (n={len(grp)})', color=palette.get(cat, 'gray'), edgecolor='white')
    axes[1].set_xlabel('Erreur de reconstruction (AE)')
    axes[1].set_ylabel('Nombre de records')
    axes[1].set_title('Score anomalie par catégorie')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Distribution des scores ML

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, title, color in zip(axes,
    ['anomaly_score_if', 'reconstruction_error', 'pca_reconstruction_error'],
    ['Isolation Forest', 'Autoencoder', 'PCA baseline'],
    ['steelblue', 'darkorange', 'mediumseagreen']
):
    if col not in scores_df.columns:
        continue
    vals = scores_df[col].dropna()
    ax.hist(vals, bins=60, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(vals.quantile(0.95), color='red', linestyle='--', linewidth=1.5,
               label=f'P95 = {vals.quantile(0.95):.3f}')
    ax.set_title(title)
    ax.set_xlabel('Score')
    ax.set_ylabel('Nombre de records')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Distribution des scores d\'anomalie', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Accord ML vs règles cliniques

Les cas en désaccord sont les plus intéressants à labelliser.

In [ ]:
if 'anomaly_consensus' in scores_df.columns and 'nihl_flag' in scores_df.columns:
    ml   = scores_df['anomaly_consensus'].fillna(0).astype(int)
    nihl = scores_df['nihl_flag'].fillna(0).astype(int)

    categories = {
        'Accord : tous normaux':    ((ml == 0) & (nihl == 0)).sum(),
        'Accord : tous anormaux':   ((ml == 1) & (nihl == 1)).sum(),
        'ML seul (règle rate)':     ((ml == 1) & (nihl == 0)).sum(),
        'Règle seule (ML rate)':    ((ml == 0) & (nihl == 1)).sum(),
    }

    for label, count in categories.items():
        print(f"  {label:35s}: {count:4d} ({100*count/len(ml):.1f}%)")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Camembert
    colors = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12']
    axes[0].pie(categories.values(), labels=categories.keys(), colors=colors,
                autopct='%1.1f%%', startangle=90, textprops={'fontsize': 8})
    axes[0].set_title('Accord ML vs règle NIHL')

    # Heatmap désaccord
    cross = pd.crosstab(
        ml.map({0: 'ML: normal', 1: 'ML: anomalie'}),
        nihl.map({0: 'Règle: normal', 1: 'Règle: NIHL'})
    )
    sns.heatmap(cross, annot=True, fmt='d', cmap='Blues', ax=axes[1],
                linewidths=0.5, cbar=False)
    axes[1].set_title('ML × Règle NIHL')

    plt.tight_layout()
    plt.show()

## 5. Profil audiométrique moyen — NIHL flaggé vs non flaggé

In [ ]:
if 'anomaly_final' in scores_df.columns:
    anom_mask = scores_df['anomaly_final'].fillna(0).astype(bool)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, ear, label in zip(axes, ['L', 'R'], ['Oreille gauche (OG)', 'Oreille droite (OD)']):
        cols  = [f'{ear}_{f}' for f in STANDARD_FREQS if f'{ear}_{f}' in feature_df.columns]
        freqs = [f for f in STANDARD_FREQS if f'{ear}_{f}' in feature_df.columns]
        if not cols:
            continue

        for mask, color, name in [
            (anom_mask,  'tomato',    f'Anomalie (n={anom_mask.sum()})'),
            (~anom_mask, 'steelblue', f'Normal   (n={(~anom_mask).sum()})'),
        ]:
            subset = feature_df.loc[mask, cols]
            mean   = subset.mean()
            std    = subset.std()
            ax.plot(freqs, mean.values, color=color, linewidth=2.5, label=name, marker='o', markersize=4)
            ax.fill_between(freqs, (mean - std).values, (mean + std).values, alpha=0.12, color=color)

        ax.axhline(25, color='gray', linestyle=':', alpha=0.5, label='Norme 25 dB HL')
        ax.set_xscale('log')
        ax.set_xticks(freqs)
        ax.set_xticklabels([f'{f}' if f < 1000 else f'{f//1000}k' for f in freqs])
        ax.invert_yaxis()
        ax.set_ylim(100, -15)
        ax.set_xlabel('Fréquence (Hz)')
        ax.set_ylabel('Seuil auditif moyen (dB HL)')
        ax.set_title(label)
        ax.legend(fontsize=9)
        ax.grid(alpha=0.25)

    plt.suptitle('Profil audiométrique moyen : Anomalie vs Normal (±1σ)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 6. Zone grise — cas prioritaires pour labellisation

Les cas avec un score entre P50 et P90 sont les plus incertains pour le modèle.  
C'est là que la labellisation apporte le plus de valeur.

In [ ]:
if 'reconstruction_error' in scores_df.columns:
    score = scores_df['reconstruction_error']
    p50, p90 = score.quantile(0.50), score.quantile(0.90)

    grey_zone = (score >= p50) & (score <= p90)
    high_conf_anomaly = score > p90
    high_conf_normal  = score < p50

    print(f"Cas haute confiance normaux  (score < P50) : {high_conf_normal.sum():4d} ({100*high_conf_normal.mean():.0f}%)")
    print(f"Zone grise           (P50–P90)            : {grey_zone.sum():4d} ({100*grey_zone.mean():.0f}%)")
    print(f"Cas haute confiance anomalies (score > P90): {high_conf_anomaly.sum():4d} ({100*high_conf_anomaly.mean():.0f}%)")

    # Répartition zone grise par fichier
    grey_by_file = scores_df[grey_zone].groupby('source_file').size()
    total_by_file = scores_df.groupby('source_file').size()
    pct_grey = (100 * grey_by_file / total_by_file).round(1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Distribution score colorée par zone
    axes[0].hist(score[high_conf_normal],  bins=40, color='steelblue',  alpha=0.7, label=f'Normal confiant (< P50)')
    axes[0].hist(score[grey_zone],          bins=40, color='orange',      alpha=0.7, label=f'Zone grise (P50–P90)')
    axes[0].hist(score[high_conf_anomaly],  bins=40, color='tomato',      alpha=0.7, label=f'Anomalie confiante (> P90)')
    axes[0].axvline(p50, color='orange', linestyle='--', linewidth=1.5)
    axes[0].axvline(p90, color='red',    linestyle='--', linewidth=1.5)
    axes[0].set_xlabel('Erreur de reconstruction')
    axes[0].set_ylabel('Nombre de records')
    axes[0].set_title('Zones de confiance du modèle')
    axes[0].legend(fontsize=8)
    axes[0].grid(alpha=0.3)

    # % zone grise par fichier
    pct_grey.plot(kind='bar', ax=axes[1], color='orange', edgecolor='white', alpha=0.85)
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right', fontsize=8)
    axes[1].set_ylabel('% de records en zone grise')
    axes[1].set_title('Zone grise par fichier source')
    axes[1].grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.show()

## 7. Top anomalies — visualisation audiogrammes

In [ ]:
N_SHOW = 6

if 'reconstruction_error' in scores_df.columns:
    top_idx = scores_df['reconstruction_error'].nlargest(N_SHOW).index
    fig, axes = plt.subplots(2, N_SHOW // 2, figsize=(14, 8))
    axes = axes.flatten()

    for i, idx in enumerate(top_idx):
        row    = df.loc[idx]
        score  = scores_df.loc[idx, 'reconstruction_error']
        cat    = VISIT_LABEL.get(row.get('visit_category'), '?')
        gender = GENDER_LABEL.get(row.get('gender'), '?')
        age    = row.get('age_at_visit', float('nan'))
        nihl   = int(scores_df.loc[idx, 'nihl_flag']) if 'nihl_flag' in scores_df.columns else '?'
        age_str = f"{age:.0f}ans" if pd.notna(age) else '?'
        title  = f"#{idx} — AE={score:.3f}\n{cat} | {gender} | {age_str} | NIHL={nihl}"
        plot_audiogram(row['dots_left'], row['dots_right'], title=title, ax=axes[i])

    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.suptitle(f'Top {N_SHOW} anomalies — erreur de reconstruction (Autoencoder)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 8. Export — 100 cas prioritaires pour labellisation audiologiste

Trois groupes complémentaires, chacun révèle un angle mort différent :

- **Règle seule** : les règles cliniques détectent une anomalie que le ML rate → angle mort du modèle
- **ML seul** : le ML détecte quelque chose que les règles ne capturent pas → nouveau pattern ou faux positif
- **Zone grise** : le modèle est incertain (P50–P90) → calibre la frontière de décision

In [ ]:
N_TOTAL = 100   # cas à envoyer à l'audiologiste

has_rules = 'nihl_flag' in scores_df.columns and 'meniere_flag' in scores_df.columns
has_ml    = 'anomaly_consensus' in scores_df.columns
has_score = 'reconstruction_error' in scores_df.columns

if has_rules and has_ml and has_score:
    score   = scores_df['reconstruction_error']
    p50, p90 = score.quantile(0.50), score.quantile(0.90)

    rule_pos = (scores_df['nihl_flag'].fillna(0) == 1) | (scores_df['meniere_flag'].fillna(0) == 1)
    ml_pos   = scores_df['anomaly_consensus'].fillna(0) == 1

    # ── Groupe 1 : Règle seule (règle=1, ML=0) ───────────────────────────────
    mask_rule_only = rule_pos & ~ml_pos
    # ── Groupe 2 : ML seul (ML=1, règle=0) ───────────────────────────────────
    mask_ml_only   = ml_pos & ~rule_pos
    # ── Groupe 3 : Zone grise (P50–P90, pas déjà dans les deux autres) ───────
    mask_grey      = (score >= p50) & (score <= p90) & ~rule_pos & ~ml_pos

    # Répartition équilibrée sur 100 cas
    n_each = N_TOTAL // 3
    n_grey = N_TOTAL - 2 * n_each   # absorbe le reste

    def sample_group(mask, n, label):
        idx = scores_df[mask].index
        s   = scores_df.loc[idx, 'reconstruction_error']
        # Tri par score décroissant pour garder les cas les plus marqués
        idx_sorted = s.sort_values(ascending=False).index
        return idx_sorted[:min(n, len(idx_sorted))], label

    idx_rule, lbl_rule = sample_group(mask_rule_only, n_each,  'regle_seule')
    idx_ml,   lbl_ml   = sample_group(mask_ml_only,   n_each,  'ml_seul')
    idx_grey, lbl_grey = sample_group(mask_grey,       n_grey,  'zone_grise')

    print(f"Règle seule disponibles  : {mask_rule_only.sum():4d}  → sélectionnés : {len(idx_rule)}")
    print(f"ML seul disponibles      : {mask_ml_only.sum():4d}  → sélectionnés : {len(idx_ml)}")
    print(f"Zone grise disponibles   : {mask_grey.sum():4d}  → sélectionnés : {len(idx_grey)}")

    all_idx    = idx_rule.append(idx_ml).append(idx_grey)
    group_col  = ([lbl_rule] * len(idx_rule) +
                  [lbl_ml]   * len(idx_ml)   +
                  [lbl_grey] * len(idx_grey))

    export = scores_df.loc[all_idx, [c for c in
        ['reconstruction_error', 'anomaly_consensus', 'nihl_flag', 'meniere_flag']
        if c in scores_df.columns]].copy()
    export['labeling_group']  = group_col
    export['patient']         = df.loc[all_idx, 'patient'].values
    export['visit_date']      = df.loc[all_idx, 'visit_date'].dt.strftime('%Y-%m-%d').values
    export['visit_category']  = df.loc[all_idx, 'visit_category'].map(VISIT_LABEL).values
    export['age_at_visit']    = df.loc[all_idx, 'age_at_visit'].round(1).values
    export['gender']          = df.loc[all_idx, 'gender'].map(GENDER_LABEL).values
    export['label']           = ''   # à remplir : normal / anomalie / douteux
    export['confidence']      = ''   # haut / moyen / faible
    export['comment']         = ''

    out_path = Path('../labels/batch_01_to_label.csv')
    out_path.parent.mkdir(exist_ok=True)
    export.to_csv(out_path)

    print(f"\nTotal exporté : {len(export)} cas → {out_path}")

    # Visualisation de la répartition
    fig, ax = plt.subplots(figsize=(7, 4))
    counts = pd.Series({'Règle seule': len(idx_rule),
                         'ML seul':    len(idx_ml),
                         'Zone grise': len(idx_grey)})
    colors = ['#e74c3c', '#3498db', '#f39c12']
    bars = ax.bar(counts.index, counts.values, color=colors, edgecolor='white', alpha=0.9)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                str(val), ha='center', fontsize=11, fontweight='bold')
    ax.set_ylabel('Nombre de cas')
    ax.set_title(f'Répartition des {len(export)} cas à labelliser')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()